# Análisis de Sentimientos en Python

## 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

%matplotlib inline

print("Bibliotecas importadas correctamente")

✓ Listo


## 2. Cargar el dataset de reseñas

In [ ]:
# Cargar el dataset
import os

if os.path.exists('reviews_procesados.csv'):
    print("Cargando archivo local...")
    df = pd.read_csv('reviews_procesados.csv')
else:
    # Subir archivo si no existe
    from google.colab import files
    print("Por favor, sube el archivo 'reviews_procesados.csv'")
    uploaded = files.upload()
    df = pd.read_csv('reviews_procesados.csv')

# Mostrar primeras filas
print("\nDataset cargado correctamente")
print(f"Dimensiones: {df.shape}")
df.head()

SUBE EL ARCHIVO 'reviews_procesados.csv' DESDE TU ORDENADOR

Haz clic en 'Elegir archivos' y selecciona:
   C:\Users\anton\Desktop\2ºGS\SGE\reviews_procesados.csv



## 3. Limpieza de datos

In [ ]:
# Features: columna de texto
# Labels: columna de sentimiento/valoración

features = df.iloc[:, 1].values  # Ajustar según la columna de texto en tu CSV
labels = df.iloc[:, 0].values     # Ajustar según la columna de valoración

print(f"Features extraídas: {len(features)} textos")
print(f"Labels extraídas: {len(labels)} etiquetas")
print(f"\nPrimeras 3 features:")
for i in range(min(3, len(features))):
    print(f"  [{i}]: {features[i][:100]}...")

## 4. Procesar los datos mediante expresiones regulares (RegEx)

In [ ]:
processed_features = []

for sentence in range(0, len(features)):
    # Eliminar todos los caracteres especiales
    processed_feature = re.sub(r'\W', ' ', str(features[sentence]))

    # Eliminar todos los caracteres individuales
    processed_feature= re.sub(r'\s+[a-zA-Z]\s+', ' ', processed_feature)

    # Eliminar caracteres individuales del inicio
    processed_feature = re.sub(r'\^[a-zA-Z]\s+', ' ', processed_feature) 

    # Sustituir múltiples espacios con un solo espacio
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # Eliminar prefijo 'b'
    processed_feature = re.sub(r'^b\s+', '', processed_feature)

    # Convertir a minúsculas
    processed_feature = processed_feature.lower()

    processed_features.append(processed_feature)

print(f"{len(processed_features)} textos procesados correctamente")
print(f"\nEjemplo de texto procesado:")
print(f"  Original: {features[0][:100]}")
print(f"  Procesado: {processed_features[0][:100]}")

## 5. Representación del texto en forma numérica - TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Stopwords en español
spanish_stopwords = ['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 
                     'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como',
                     'más', 'pero', 'sus', 'le', 'ya', 'o', 'fue', 'este', 'ha', 'sí',
                     'porque', 'esta', 'son', 'entre', 'está', 'cuando', 'muy', 'sin',
                     'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'han', 'quien',
                     'están', 'estado', 'desde', 'todo', 'nos', 'durante', 'estados',
                     'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'fueron', 'ese',
                     'eso', 'había', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos',
                     'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa',
                     'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'sea', 'poco',
                     'ella', 'estar', 'haber', 'estas', 'estaba', 'estamos', 'algunas',
                     'algo', 'nosotros']

# Configuración: max_features=2500, min_df=7, max_df=0.8
vectorizer = TfidfVectorizer(max_features=2500, min_df=7, max_df=0.8, stop_words=spanish_stopwords)
processed_features = vectorizer.fit_transform(processed_features).toarray()

print(f"Vectorización TF-IDF completada correctamente")
print(f"  Shape: {processed_features.shape}")
print(f"  Features: {processed_features.shape[0]} textos")
print(f"  Vocabulario: {processed_features.shape[1]} palabras únicas")

## 6. Dividir los datos en conjuntos de entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

# División 80/20 (test_size=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(processed_features, labels, test_size=0.2, random_state=0)

print(f"División train/test completada correctamente")
print(f"  Training set: {X_train.shape[0]} muestras")
print(f"  Test set: {X_test.shape[0]} muestras")
print(f"  Ratio: {X_train.shape[0]/len(labels)*100:.1f}% / {X_test.shape[0]/len(labels)*100:.1f}%")

## 7. Entrenar el modelo (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

print("=" * 80)
print("PROBANDO MULTIPLES CONFIGURACIONES PARA MAXIMIZAR PRECISION")
print("=" * 80)

# Diccionario para almacenar resultados
resultados = {}

# 1. Random Forest - Configuración con 200 árboles
print("\n[1/6] Random Forest (200 árboles)...")
rf_200 = RandomForestClassifier(n_estimators=200, random_state=0)
rf_200.fit(X_train, y_train)
pred_rf_200 = rf_200.predict(X_test)
acc_rf_200 = accuracy_score(y_test, pred_rf_200)
resultados['Random Forest (200 árboles)'] = acc_rf_200
print(f"      Accuracy: {acc_rf_200 * 100:.2f}%")

# 2. Random Forest - Más árboles
print("\n[2/6] Random Forest (500 árboles)...")
rf_500 = RandomForestClassifier(n_estimators=500, random_state=0)
rf_500.fit(X_train, y_train)
pred_rf_500 = rf_500.predict(X_test)
acc_rf_500 = accuracy_score(y_test, pred_rf_500)
resultados['Random Forest (500 árboles)'] = acc_rf_500
print(f"      Accuracy: {acc_rf_500 * 100:.2f}%")

# 3. Logistic Regression
print("\n[3/6] Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=0)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, pred_lr)
resultados['Logistic Regression'] = acc_lr
print(f"      Accuracy: {acc_lr * 100:.2f}%")

# 4. Naive Bayes
print("\n[4/6] Multinomial Naive Bayes...")
nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test, pred_nb)
resultados['Naive Bayes'] = acc_nb
print(f"      Accuracy: {acc_nb * 100:.2f}%")

# 5. Linear SVM
print("\n[5/6] Linear SVM...")
svm = LinearSVC(max_iter=1000, random_state=0)
svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, pred_svm)
resultados['Linear SVM'] = acc_svm
print(f"      Accuracy: {acc_svm * 100:.2f}%")

# 6. Random Forest optimizado
print("\n[6/6] Random Forest Optimizado (300 árboles, max_depth=20)...")
rf_opt = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=0)
rf_opt.fit(X_train, y_train)
pred_rf_opt = rf_opt.predict(X_test)
acc_rf_opt = accuracy_score(y_test, pred_rf_opt)
resultados['Random Forest Optimizado'] = acc_rf_opt
print(f"      Accuracy: {acc_rf_opt * 100:.2f}%")

# Encontrar el mejor modelo
mejor_modelo = max(resultados, key=resultados.get)
mejor_accuracy = resultados[mejor_modelo]

print("\n" + "=" * 80)
print("RESUMEN DE RESULTADOS")
print("=" * 80)
for modelo, acc in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    marca = "MEJOR" if modelo == mejor_modelo else "  "
    print(f"{marca} {modelo:35s} -> {acc*100:6.2f}%")

print("\n" + "=" * 80)
print(f"MEJOR MODELO: {mejor_modelo}")
print(f"   Precisión: {mejor_accuracy * 100:.2f}%")
print("=" * 80)

# Guardar el mejor modelo para evaluación detallada
if mejor_modelo == 'Random Forest (200 árboles)':
    text_classifier = rf_200
elif mejor_modelo == 'Random Forest (500 árboles)':
    text_classifier = rf_500
elif mejor_modelo == 'Logistic Regression':
    text_classifier = lr
elif mejor_modelo == 'Naive Bayes':
    text_classifier = nb
elif mejor_modelo == 'Linear SVM':
    text_classifier = svm
else:
    text_classifier = rf_opt

## 8. Realizar predicciones y evaluar el modelo

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Hacer predicciones con el mejor modelo
predictions = text_classifier.predict(X_test)

# Evaluación detallada del mejor modelo
print("=" * 80)
print(f"EVALUACION DETALLADA DEL MEJOR MODELO: {mejor_modelo}")
print("=" * 80)

print("\nMatriz de Confusion:")
cm = confusion_matrix(y_test, predictions)
print(cm)

print("\nReporte de Clasificacion:")
print(classification_report(y_test, predictions, target_names=['Negativo', 'Positivo']))

accuracy = accuracy_score(y_test, predictions)
print(f"\n{'='*80}")
print(f"RESULTADO FINAL")
print(f"{'='*80}")
print(f"Accuracy Score: {accuracy:.4f}")
print(f"Porcentaje de acierto: {accuracy * 100:.2f}%")
print(f"Total test samples: {len(y_test)}")
print(f"Predicciones correctas: {int(accuracy * len(y_test))}")
print(f"Predicciones incorrectas: {int((1-accuracy) * len(y_test))}")
print("=" * 80)

## 9. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Gráfico de comparación de modelos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Gráfico de barras con precisión de cada modelo
modelos_nombres = list(resultados.keys())
precisiones = [resultados[m] * 100 for m in modelos_nombres]
colores = ['#2ecc71' if m == mejor_modelo else '#3498db' for m in modelos_nombres]

ax1.barh(modelos_nombres, precisiones, color=colores)
ax1.set_xlabel('Precisión (%)', fontsize=12)
ax1.set_title('Comparación de Precisión por Modelo', fontsize=14, fontweight='bold')
ax1.set_xlim([0, 100])
for i, v in enumerate(precisiones):
    ax1.text(v + 1, i, f'{v:.2f}%', va='center', fontsize=10)

# 2. Matriz de confusión del mejor modelo
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'])
ax2.set_title(f'Matriz de Confusión - {mejor_modelo}', fontsize=14, fontweight='bold')
ax2.set_ylabel('Valor Real', fontsize=12)
ax2.set_xlabel('Predicción', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\nVisualizaciones generadas para {mejor_modelo}")